# Brain Tumor MRI Classification with Transfer Learning

This portfolio notebook compares **ResNet50, DenseNet121 and EfficientNetB0** for four-class brain MRI image classification using transfer learning.

**Classes:** glioma, meningioma, normal, pituitary  
**Dataset:** Crystal Clean: Brain Tumors MRI Dataset (Kaggle)  
**Task:** multi-class image classification  

> **Academic/educational prototype only — not for clinical diagnosis or patient care.**

The workflow covers dataset inspection, deterministic image-level splitting, preprocessing, augmentation, class weighting, two-stage transfer learning, held-out test evaluation, confusion matrices, ROC analysis and Grad-CAM visual explanation.


# 1. Setup

Run this notebook in Google Colab with a GPU runtime.

In [ ]:
# Install KaggleHub if needed.
try:
    import kagglehub
except ImportError:
    !pip install -q kagglehub
    import kagglehub

import os
import random
import shutil
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers
from tensorflow.keras.applications import ResNet50, DenseNet121, EfficientNetB0
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("TensorFlow version:", tf.__version__)
print("Available GPU:", tf.config.list_physical_devices("GPU"))

# 2. Experiment configuration

The settings below are used consistently for all three models so the comparison is fair.

In [ ]:
SEED = 7023
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Project dataset: Crystal Clean: Brain Tumors MRI Dataset
KAGGLE_DATASET_ID = "mohammadhossein77/brain-tumors-dataset"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Increase these for stronger results if you have enough Colab GPU time.
HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 5
HEAD_LR = 1e-3
FINE_TUNE_LR = 1e-5

MAX_IMAGES_PER_CLASS = None

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
WORK_DIR = BASE_DIR / "brain_tumor_mri_work"
FLAT_DIR = WORK_DIR / "flat_four_classes"
SPLIT_DIR = WORK_DIR / "train_val_test"
FIG_DIR = WORK_DIR / "figures"
WEIGHTS_DIR = WORK_DIR / "weights"
RESULTS_DIR = WORK_DIR / "results"

for folder in [WORK_DIR, FLAT_DIR, SPLIT_DIR, FIG_DIR, WEIGHTS_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Working directory:", WORK_DIR)

# 3. Download and inspect the Kaggle dataset

The dataset is downloaded directly from Kaggle using `kagglehub`.

In [ ]:
print("Downloading Crystal Clean: Brain Tumors MRI Dataset from Kaggle...")
dataset_path = Path(kagglehub.dataset_download(KAGGLE_DATASET_ID))
print(f"Dataset downloaded to: {dataset_path}")

print("\nFiles and folders available:")
for item in sorted(dataset_path.iterdir()):
    if item.is_file():
        size_mb = item.stat().st_size / 1e6
        print(f"  FILE   {item.name:<45} {size_mb:>8.2f} MB")
    else:
        print(f"  FOLDER {item.name}")

In [ ]:
def print_folder_tree(root, max_depth=4):
    # Compact folder tree display for Colab.
    root = Path(root)
    print(root)
    for current_root, dirs, files in os.walk(root):
        current_root = Path(current_root)
        depth = len(current_root.relative_to(root).parts)
        if depth >= max_depth:
            dirs[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}├── {current_root.name}/ ({len(files)} files)")
        for folder_name in sorted(dirs)[:12]:
            print(f"{indent}│   └── {folder_name}/")

print_folder_tree(dataset_path, max_depth=4)

# 4_ Fix the nested class-folder structure

Keras needs one folder per class. This cell detects the original folders and creates:

```text
flat_four_classes/
├── glioma/
├── meningioma/
├── normal/
└── pituitary/
```

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

CLASS_FOLDER_MAP = {
    "normal": ["Normal", "normal", "notumor", "no_tumor", "no tumor"],
    "glioma": ["glioma_tumor", "glioma", "Glioma"],
    "meningioma": ["meningioma_tumor", "meningioma", "Meningioma"],
    "pituitary": ["pituitary_tumor", "pituitary", "Pituitary"]
}

def clean_name(text):
    return text.lower().replace(" ", "").replace("_", "").replace("-", "")

def locate_class_folder(root, possible_names):
    targets = {clean_name(name) for name in possible_names}
    matches = []
    for path in Path(root).rglob("*"):
        if path.is_dir() and clean_name(path.name) in targets:
            count = sum(1 for f in path.rglob("*") if f.suffix.lower() in IMAGE_EXTENSIONS)
            if count > 0:
                matches.append((path, count))
    if not matches:
        return None
    matches.sort(key=lambda x: x[1], reverse=True)
    return matches[0][0]

def link_or_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)

def build_clean_class_folders(source_root, target_root, max_per_class=None):
    source_root = Path(source_root)
    target_root = Path(target_root)
    if target_root.exists():
        shutil.rmtree(target_root)
    target_root.mkdir(parents=True, exist_ok=True)

    report_rows = []
    for clean_class, folder_candidates in CLASS_FOLDER_MAP.items():
        found_folder = locate_class_folder(source_root, folder_candidates)
        if found_folder is None:
            raise FileNotFoundError(f"Could not locate folder for class: {clean_class}")

        images = sorted([f for f in found_folder.rglob("*") if f.suffix.lower() in IMAGE_EXTENSIONS])
        random.Random(SEED).shuffle(images)
        if max_per_class is not None:
            images = images[:max_per_class]

        class_target = target_root / clean_class
        class_target.mkdir(parents=True, exist_ok=True)

        for i, src in enumerate(images):
            dst = class_target / f"{clean_class}_{i:06d}{src.suffix.lower()}"
            link_or_copy(src, dst)

        report_rows.append({
            "class_name": clean_class,
            "original_folder_found": str(found_folder),
            "image_count": len(images)
        })

    return pd.DataFrame(report_rows)

source_report = build_clean_class_folders(dataset_path, FLAT_DIR, MAX_IMAGES_PER_CLASS)
display(source_report)
print("Clean class directory:", FLAT_DIR)

# 5. Dataset exploration

This section summarises the number of classes, image counts, image dimensions, class distribution and representative samples.

In [ ]:
def count_images_by_class(root_dir):
    rows = []
    for class_dir in sorted([p for p in Path(root_dir).iterdir() if p.is_dir()]):
        files = [f for f in class_dir.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS]
        rows.append({"class_name": class_dir.name, "image_count": len(files)})
    return pd.DataFrame(rows)

class_count_df = count_images_by_class(FLAT_DIR)
print("Number of classes:", len(class_count_df))
display(class_count_df)

dimension_rows = []
for class_dir in sorted([p for p in FLAT_DIR.iterdir() if p.is_dir()]):
    sample_files = sorted([f for f in class_dir.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS])[:25]
    for file_path in sample_files:
        with Image.open(file_path) as img:
            dimension_rows.append({
                "class_name": class_dir.name,
                "width": img.size[0],
                "height": img.size[1],
                "colour_mode": img.mode
            })

dimension_df = pd.DataFrame(dimension_rows)
display(dimension_df.groupby(["class_name", "width", "height", "colour_mode"]).size().reset_index(name="sample_count"))

## 5.1 Dataset summary

In [ ]:
from PIL import Image
from pathlib import Path
import pandas as pd

def folder_size_mb(folder):
    total_bytes = 0
    for path in Path(folder).rglob("*"):
        if path.is_file():
            total_bytes += path.stat().st_size
    return total_bytes / (1024 * 1024)

image_extensions = [".jpg", ".jpeg", ".png", ".bmp"]

class_rows = []
for class_folder in sorted(FLAT_DIR.iterdir()):
    if class_folder.is_dir():
        image_files = [
            p for p in class_folder.rglob("*")
            if p.suffix.lower() in image_extensions
        ]
        class_rows.append({
            "Class Name": class_folder.name,
            "Number of Images": len(image_files)
        })

class_count_table = pd.DataFrame(class_rows)

sample_image_path = next(
    p for p in FLAT_DIR.rglob("*")
    if p.suffix.lower() in image_extensions
)

with Image.open(sample_image_path) as img:
    original_width, original_height = img.size
    original_mode = img.mode

dataset_summary = pd.DataFrame({
    "Item": [
        "Dataset Name",
        "Dataset Link",
        "Dataset Size (MB)",
        "Original Sample Image Dimensions",
        "Original Colour Mode",
        "Model Input Image Dimensions",
        "Number of Classes",
        "Total Number of Images"
    ],
    "Value": [
        "Brain Tumors MRI Dataset",
        "https://www.kaggle.com/datasets/mohammadhossein77/brain-tumors-dataset",
        f"{folder_size_mb(FLAT_DIR):.2f} MB",
        f"{original_width} x {original_height}",
        original_mode,
        f"{IMG_SIZE[0]} x {IMG_SIZE[1]} x 3",
        NUM_CLASSES,
        class_count_table["Number of Images"].sum()
    ]
})

display(dataset_summary)
display(class_count_table)

dataset_summary.to_csv(RESULTS_DIR / "dataset_summary.csv", index=False)
class_count_table.to_csv(RESULTS_DIR / "class_counts.csv", index=False)

## 5_3 (P) Plot. Class Distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(class_count_df["class_name"], class_count_df["image_count"])
plt.title("Class distribution before splitting")
plt.xlabel("MRI class")
plt.ylabel("Number of images")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_full_dataset_class_distribution.png", dpi=150)
plt.show()

## 5.2 Sample images

In [ ]:
def show_sample_grid(root_dir, images_per_class=4):
    class_dirs = sorted([p for p in Path(root_dir).iterdir() if p.is_dir()])
    fig, axes = plt.subplots(len(class_dirs), images_per_class, figsize=(12, 3 * len(class_dirs)))

    for row, class_dir in enumerate(class_dirs):
        files = [f for f in class_dir.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS]
        random.Random(SEED + row).shuffle(files)
        for col, file_path in enumerate(files[:images_per_class]):
            image = Image.open(file_path).convert("RGB")
            axes[row, col].imshow(image)
            axes[row, col].set_title(class_dir.name)
            axes[row, col].axis("off")

    plt.suptitle("Sample MRI images from each class", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "02_sample_mri_images.png", dpi=150)
    plt.show()

show_sample_grid(FLAT_DIR, images_per_class=4)

# 6. Stratified train / validation / test split

The split is made class-by-class so each partition has all four labels.

In [ ]:
def reset_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def make_split_folders(flat_root, split_root):
    reset_dir(split_root)
    records = []

    for class_dir in sorted([p for p in Path(flat_root).iterdir() if p.is_dir()]):
        files = [f for f in class_dir.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS]
        random.Random(SEED).shuffle(files)

        n_total = len(files)
        n_train = int(n_total * TRAIN_RATIO)
        n_val = int(n_total * VAL_RATIO)

        split_files = {
            "train": files[:n_train],
            "validation": files[n_train:n_train + n_val],
            "test": files[n_train + n_val:]
        }

        for split_name, selected_files in split_files.items():
            target_class_dir = Path(split_root) / split_name / class_dir.name
            target_class_dir.mkdir(parents=True, exist_ok=True)
            for src in selected_files:
                link_or_copy(src, target_class_dir / src.name)
            records.append({
                "split": split_name,
                "class_name": class_dir.name,
                "image_count": len(selected_files)
            })

    return pd.DataFrame(records)

split_df = make_split_folders(FLAT_DIR, SPLIT_DIR)
split_counts_table = split_df.pivot(index="class_name", columns="split", values="image_count").reset_index()
display(split_counts_table)
split_counts_table.to_csv(RESULTS_DIR / "split_counts.csv", index=False)

## 6.1 Split distribution

In [ ]:
split_pivot = split_df.pivot(index="class_name", columns="split", values="image_count")
split_pivot.plot(kind="bar", figsize=(9, 5))
plt.title("Class distribution after train/validation/test splitting")
plt.xlabel("MRI class")
plt.ylabel("Number of images")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "03_train_validation_test_distribution.png", dpi=150)
plt.show()

# 7. TensorFlow image encoding pipeline

The images are loaded as 224 × 224 RGB tensors with integer labels. The model applies its own preprocessing layer.

In [ ]:
def load_image_dataset(folder, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        folder,
        labels="inferred",
        label_mode="int",
        color_mode="rgb",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED
    )

train_ds = load_image_dataset(SPLIT_DIR / "train", shuffle=True)
val_ds = load_image_dataset(SPLIT_DIR / "validation", shuffle=False)
test_ds = load_image_dataset(SPLIT_DIR / "test", shuffle=False)

class_names = train_ds.class_names
print("TensorFlow class order:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# Class weights reduce bias towards larger classes.
train_counts = split_df[split_df["split"] == "train"].set_index("class_name").loc[class_names]["image_count"]
total_train = train_counts.sum()
class_weight = {i: total_train / (NUM_CLASSES * count) for i, count in enumerate(train_counts)}

print("Class weights:")
for i, weight in class_weight.items():
    print(f"{i}: {class_names[i]:<12} {weight:.3f}")

## 7_1 (P) Data Pre-processing & Encoding

In [ ]:
for image_batch, label_batch in train_ds.take(1):
    print("Image batch shape:", image_batch.shape)
    print("Label batch shape:", label_batch.shape)
    print("Image data type:", image_batch.dtype)
    print("Label data type:", label_batch.dtype)
    print("Pixel value range before model-specific preprocessing:",
          float(tf.reduce_min(image_batch)),
          "to",
          float(tf.reduce_max(image_batch)))

## 7_2 (P) Visual comparison of preprocessing techniques

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
from pathlib import Path
from tensorflow.keras import layers

# Choose class and image index
CLASS_NAME = "meningioma"      # change to: glioma, meningioma, normal, pituitary
SAMPLE_INDEX = 10

TRAIN_FOLDER = SPLIT_DIR / "train" / CLASS_NAME

image_extensions = ["*.jpg", "*.jpeg", "*.png"]

all_images = []
for ext in image_extensions:
    all_images.extend(list(Path(TRAIN_FOLDER).rglob(ext)))

all_images = sorted(all_images)

if len(all_images) == 0:
    raise FileNotFoundError(f"No image file found in {TRAIN_FOLDER}")

sample_path = all_images[SAMPLE_INDEX % len(all_images)]

print("Class selected:", CLASS_NAME)
print("Number of available images in class:", len(all_images))
print("Sample index used:", SAMPLE_INDEX)
print("Sample image used:", sample_path)

if sample_path is None:
    raise FileNotFoundError(f"No image file found in {TRAIN_FOLDER}")

print("Sample image used:", sample_path)

# Load original image
raw_image = tf.io.read_file(str(sample_path))
image = tf.image.decode_image(raw_image, channels=3, expand_animations=False)
image = tf.image.convert_image_dtype(image, tf.float32)

# Resize to model input size
resized_image = tf.image.resize(image, IMG_SIZE)

# Add batch dimension for augmentation layers
image_batch = tf.expand_dims(resized_image, axis=0)

# Individual preprocessing / augmentation examples
flipped_image = tf.image.flip_left_right(resized_image)

rotated_image = layers.RandomRotation(0.04)(image_batch, training=True)[0]
zoomed_image = layers.RandomZoom(0.10)(image_batch, training=True)[0]
contrast_image = layers.RandomContrast(0.10)(image_batch, training=True)[0]

# Model-specific preprocessing example using ResNet50 preprocessing
resnet_ready = resized_image * 255.0
resnet_preprocessed = tf.keras.applications.resnet50.preprocess_input(resnet_ready)

# For display only: scale preprocessed image back to 0-1
resnet_display = resnet_preprocessed.numpy()
resnet_display = (resnet_display - resnet_display.min()) / (
    resnet_display.max() - resnet_display.min()
)

# Prepare images and titles
images_to_show = [
    image,
    resized_image,
    resized_image,
    flipped_image,
    rotated_image,
    zoomed_image,
    contrast_image,
    resnet_display
]

titles = [
    "Original MRI",
    "Resized 224×224",
    "RGB Image",
    "Random Flip",
    "Random Rotation",
    "Random Zoom",
    "Random Contrast",
    "ResNet50 Preprocessing\n(display scaled)"
]

# Plot comparison
plt.figure(figsize=(16, 8))

for i, (img, title) in enumerate(zip(images_to_show, titles)):
    plt.subplot(2, 4, i + 1)

    img_np = img.numpy() if hasattr(img, "numpy") else img
    img_np = np.clip(img_np, 0, 1)

    plt.imshow(img_np)
    plt.title(title, fontsize=11)
    plt.axis("off")

plt.suptitle(
    "Visual Comparison of Preprocessing and Augmentation Techniques Applied to a Brain MRI Image",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()

output_path = FIG_DIR / "visual_preprocessing_comparison_mri.png"
plt.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved visual preprocessing comparison to:", output_path)

# 8_ Architecture design

Each architecture uses the same classification head:

```text
Input MRI Image 224x224x3
          ↓
Data Augmentation
          ↓
Model-specific Preprocessing
          ↓
Pretrained CNN Backbone
          ↓
Global Average Pooling
          ↓
Dropout 0.35
          ↓
Dense Layer 128 ReLU
          ↓
Dropout 0.25
          ↓
Softmax Output Layer, 4 Classes
```



## 8_1 (P) Only the backbone changes: ResNet50, DenseNet121, or EfficientNetB0.

In [ ]:
MODEL_REGISTRY = {
    "ResNet50": {
        "constructor": ResNet50,
        "preprocess": resnet_preprocess,
        "fine_tune_ratio": 0.75
    },
    "DenseNet121": {
        "constructor": DenseNet121,
        "preprocess": densenet_preprocess,
        "fine_tune_ratio": 0.75
    },
    "EfficientNetB0": {
        "constructor": EfficientNetB0,
        "preprocess": efficientnet_preprocess,
        "fine_tune_ratio": 0.75
    }
}

def make_augmentation():
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.04),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10)
    ], name="mri_augmentation")

def build_model(model_name):
    spec = MODEL_REGISTRY[model_name]
    base = spec["constructor"](
        weights="imagenet",
        include_top=False,
        input_shape=IMG_SIZE + (3,)
    )
    base.trainable = False
    base._name = f"{model_name}_base"

    inputs = keras.Input(shape=IMG_SIZE + (3,), name="mri_input")
    x = make_augmentation()(inputs)
    x = layers.Lambda(spec["preprocess"], name=f"{model_name}_preprocess")(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_pool")(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(128, activation="relu", name="feature_projection")(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="class_output")(x)

    model = keras.Model(inputs, outputs, name=f"CN7023_{model_name}")
    return model, base

def print_parameter_summary(model, model_name):
    trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
    total_params = model.count_params()
    print(f"\n{model_name}")
    print("-" * 50)
    print(f"Total parameters:     {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters:    {total_params - trainable_params:,}")

## 8_2 (P) ResNet50 Architecture

In [ ]:
from tensorflow.keras.utils import plot_model

# Choose the best model for the architecture slide.
# If summary_df already exists, use the best-performing model.
# Otherwise, use ResNet50 as the proposed final model.
try:
    best_model_name = summary_df.iloc[0]["Model"]
except NameError:
    best_model_name = "ResNet50"

architecture_model, _ = build_model(best_model_name)

architecture_path = FIG_DIR / f"{best_model_name}_architecture.png"

plot_model(
    architecture_model,
    to_file=str(architecture_path),
    show_shapes=True,
    show_layer_names=True,
    expand_nested=True,
    dpi=120
)

print("Saved architecture diagram to:", architecture_path)
architecture_model.summary()

# 9_ (P) Training and evaluation functions

Training uses two phases: head training followed by fine-tuning. The same process is used for all models.

In [ ]:
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return merged

def train_one_model(model_name):
    print("\n" + "=" * 80)
    print("Training:", model_name)
    print("=" * 80)

    model, base = build_model(model_name)
    print_parameter_summary(model, model_name)

    checkpoint = WEIGHTS_DIR / f"{model_name}_best.weights.h5"

    cb = [
        callbacks.ModelCheckpoint(
            filepath=str(checkpoint),
            monitor="val_accuracy",
            save_best_only=True,
            save_weights_only=True,
            mode="max",
            verbose=1
        ),
        callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=3,
            mode="max",
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1
        )
    ]

    model.compile(
        optimizer=optimizers.Adam(learning_rate=HEAD_LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    print("\nPhase 1: train classification head")
    h1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        class_weight=class_weight,
        callbacks=cb,
        verbose=1
    )

    base.trainable = True
    cut = int(len(base.layers) * MODEL_REGISTRY[model_name]["fine_tune_ratio"])
    for layer in base.layers[:cut]:
        layer.trainable = False

    model.compile(
        optimizer=optimizers.Adam(learning_rate=FINE_TUNE_LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    print("\nPhase 2: fine-tune top part of pretrained backbone")
    h2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FINE_TUNE_EPOCHS,
        class_weight=class_weight,
        callbacks=cb,
        verbose=1
    )

    if checkpoint.exists():
        model.load_weights(str(checkpoint))

    history = merge_histories(h1, h2)
    return model, history, checkpoint

def get_true_labels(dataset):
    labels = []
    for _, batch_labels in dataset:
        labels.extend(batch_labels.numpy())
    return np.array(labels)

def evaluate_model(model, model_name):
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    probabilities = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)
    y_true = get_true_labels(test_ds)

    print(f"\n{model_name} test accuracy: {test_acc * 100:.2f}%")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    return {
        "model": model_name,
        "test_loss": test_loss,
        "test_accuracy_percent": test_acc * 100,
        "y_true": y_true,
        "y_pred": y_pred,
        "probabilities": probabilities
    }

# 10_ Train ResNet50, DenseNet121 and EfficientNetB0

This is the main training cell. It will take time because all three architectures are trained and compared.

In [ ]:
import time
from sklearn.metrics import f1_score

model_names = ["ResNet50", "DenseNet121", "EfficientNetB0"]

histories = {}
results = {}
weight_paths = {}
training_times = {}

for model_name in model_names:
    start_time = time.time()

    model, history, checkpoint = train_one_model(model_name)

    end_time = time.time()
    elapsed_seconds = end_time - start_time

    histories[model_name] = history
    weight_paths[model_name] = checkpoint
    training_times[model_name] = elapsed_seconds
    results[model_name] = evaluate_model(model, model_name)

    # Clear memory before the next model.
    del model
    gc.collect()
    tf.keras.backend.clear_session()

model_notes = {
    "ResNet50": "Residual skip connections + transfer learning",
    "DenseNet121": "Dense feature reuse + transfer learning",
    "EfficientNetB0": "Efficient compound scaling + transfer learning"
}

summary_df = pd.DataFrame([
    {
        "Model": name,
        "Test Accuracy (%)": results[name]["test_accuracy_percent"],
        "Test Loss": results[name]["test_loss"],
        "F1-Score": f1_score(
            results[name]["y_true"],
            results[name]["y_pred"],
            average="weighted"
        ),
        "Training Time (s)": training_times.get(name, np.nan),
        "Training Time (min)": training_times.get(name, np.nan) / 60,
        "Notes": model_notes.get(name, "Transfer learning model")
    }
    for name in model_names
]).sort_values("Test Accuracy (%)", ascending=False)

display(summary_df)
summary_df.to_csv(RESULTS_DIR / "model_metrics.csv", index=False)

## 10_1 (P) clean table Network Training Options. S5

In [ ]:
training_options_table = pd.DataFrame({
    "Training Option": [
        "Input Image Size",
        "Batch Size",
        "Number of Classes",
        "Training / Validation / Test Split",
        "Phase 1 Epochs",
        "Phase 1 Learning Rate",
        "Phase 2 Fine-tuning Epochs",
        "Phase 2 Fine-tuning Learning Rate",
        "Optimizer",
        "Loss Function",
        "Dropout Ratios",
        "Class Imbalance Handling",
        "Callbacks"
    ],
    "Value": [
        f"{IMG_SIZE[0]} x {IMG_SIZE[1]} x 3",
        BATCH_SIZE,
        NUM_CLASSES,
        f"{int(TRAIN_RATIO*100)}% / {int(VAL_RATIO*100)}% / {int(TEST_RATIO*100)}%",
        HEAD_EPOCHS,
        HEAD_LR,
        FINE_TUNE_EPOCHS,
        FINE_TUNE_LR,
        "Adam",
        "Sparse Categorical Crossentropy",
        "0.35 and 0.25",
        "Class weights",
        "ModelCheckpoint, EarlyStopping, ReduceLROnPlateau"
    ]
})

display(training_options_table)
training_options_table.to_csv(RESULTS_DIR / "training_options.csv", index=False)

# 11_ Accuracy curves

## 11_1 (P) Accuracy and Loss over Epochs

The following graphs show how each model learned over time.  
The accuracy graph compares training accuracy and validation accuracy across epochs.  
The loss graph compares training loss and validation loss across epochs.

A good model should show increasing accuracy and decreasing loss. If validation accuracy stops improving while training accuracy keeps increasing, this may indicate overfitting.

In [ ]:
def plot_accuracy_and_loss(history, model_name):
    train_acc = np.array(history["accuracy"]) * 100
    val_acc = np.array(history["val_accuracy"]) * 100
    train_loss = history["loss"]
    val_loss = history["val_loss"]

    test_acc = results[model_name]["test_accuracy_percent"]

    epochs = range(1, len(train_acc) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_acc, label="Training Accuracy")
    plt.plot(epochs, val_acc, label="Validation Accuracy")
    plt.axhline(
        y=test_acc,
        linestyle="--",
        label=f"Final Test Accuracy ({test_acc:.2f}%)"
    )
    plt.title(f"{model_name} Accuracy over Epochs")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_loss, label="Training Loss")
    plt.plot(epochs, val_loss, label="Validation Loss")
    plt.title(f"{model_name} Loss over Epochs")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.suptitle(f"{model_name}: Accuracy and Loss over Epochs", fontweight="bold")
    plt.tight_layout()

    output_path = FIG_DIR / f"{model_name}_accuracy_loss_epochs.png"
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Saved graph for {model_name} to:", output_path)


for model_name, history in histories.items():
    plot_accuracy_and_loss(history, model_name)

# 12. Confusion matrices

The confusion matrix is used to identify class-wise performance and misclassification patterns.

In [ ]:
def plot_confusion_matrix_for_model(model_name):
    y_true = results[model_name]["y_true"]
    y_pred = results[model_name]["y_pred"]
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.title(f"Confusion matrix: {model_name}")
    plt.colorbar()

    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=35, ha="right")
    plt.yticks(ticks, class_names)
    plt.xlabel("Predicted class")
    plt.ylabel("True class")

    threshold = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            colour = "white" if cm[i, j] > threshold else "black"
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color=colour)

    plt.tight_layout()
    plt.savefig(FIG_DIR / f"confusion_matrix_{model_name}.png", dpi=150)
    plt.show()

for name in model_names:
    plot_confusion_matrix_for_model(name)

# 13. Final model comparison

This bar chart supports the critical-analysis section by showing which architecture achieved the best held-out test accuracy.

In [ ]:

comparison_display = summary_df[
    [
        "Model",
        "Test Accuracy (%)",
        "F1-Score",
        "Training Time (min)",
        "Notes"
    ]
].copy()

comparison_display["Test Accuracy (%)"] = comparison_display["Test Accuracy (%)"].round(2)
comparison_display["F1-Score"] = comparison_display["F1-Score"].round(4)
comparison_display["Training Time (min)"] = comparison_display["Training Time (min)"].round(2)

display(comparison_display)
##
plt.figure(figsize=(8, 5))
plt.bar(summary_df["Model"], summary_df["Test Accuracy (%)"])
plt.title("Comparison of ResNet50, DenseNet121 and EfficientNetB0")
plt.xlabel("Model architecture")
plt.ylabel("Test accuracy (%)")
plt.ylim(max(0, summary_df["Test Accuracy (%)"].min() - 5), 100)
plt.grid(axis="y", alpha=0.3)

for i, value in enumerate(summary_df["Test Accuracy (%)"]):
    plt.text(i, value + 0.3, f"{value:.2f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(FIG_DIR / "model_accuracy_comparison.png", dpi=150)
plt.show()

best_model = summary_df.iloc[0]["Model"]
print("Best model based on test accuracy:", best_model)

In [ ]:
display(summary_df)

## 13.1 Per-class precision, recall and F1-score

This graph shows the precision, recall and F1-score for each MRI class using the best-performing model.

Precision shows how many predicted examples for a class were correct.  
Recall shows how many real examples of that class were correctly detected.  
F1-score combines precision and recall into one balanced metric.

This is useful for medical image classification because a model may have high overall accuracy but still perform weaker on one tumour class.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def plot_per_class_metrics(y_true, y_pred, class_names, model_name):
    """
    Plot per-class precision, recall and F1-score for one trained model.
    This gives a class-wise view of model performance.
    """

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=range(len(class_names)),
        zero_division=0
    )

    metrics_df = pd.DataFrame({
        "Class": class_names,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Support": support
    })

    display(metrics_df)

    # Bar chart for precision, recall and F1-score
    plot_df = metrics_df.set_index("Class")[["Precision", "Recall", "F1-Score"]]

    ax = plot_df.plot(
        kind="bar",
        figsize=(10, 5),
        width=0.8
    )

    ax.set_title(f"{model_name}: Per-Class Precision, Recall and F1-Score")
    ax.set_xlabel("Class")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(title="Metrics")
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=0) # Changed rotation to 0 for horizontal labels
    plt.tight_layout()


    output_path = FIG_DIR / f"{model_name}_per_class_precision_recall_f1.png"
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("Per-class metrics figure saved to:", output_path)

    return metrics_df
# Select the best model from the final comparison table
best_model_name = summary_df.sort_values(
    by="Test Accuracy (%)",
    ascending=False
).iloc[0]["Model"]

print("Best model selected for per-class metrics:", best_model_name)

best_y_true = results[best_model_name]["y_true"]
best_y_pred = results[best_model_name]["y_pred"]

best_model_class_metrics = plot_per_class_metrics(
    y_true=best_y_true,
    y_pred=best_y_pred,
    class_names=class_names,
    model_name=best_model_name
)

best_model_class_metrics.to_csv(
    RESULTS_DIR / "best_model_per_class_metrics.csv",
    index=False
)


## 13.2 ROC curves for each class

The ROC curve shows the trade-off between true positive rate and false positive rate for each class.

For this multi-class MRI classification task, ROC is calculated using a one-vs-rest approach. This means each class is compared against all other classes. The AUC value summarises the model's ability to separate that class from the remaining classes.

A curve closer to the top-left corner and an AUC closer to 1.00 indicates stronger class discrimination.

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

def plot_multiclass_roc_curves(y_true, y_score, class_names, model_name):
    """
    Plot ROC curves for each class using a one-vs-rest approach.

    y_true  : true class labels, shape = (n_samples,)
    y_score : predicted probability scores, shape = (n_samples, n_classes)
    """

    n_classes = len(class_names)

    # Convert class labels into one-hot format for ROC calculation
    y_true_bin = label_binarize(
        y_true,
        classes=list(range(n_classes))
    )

    plt.figure(figsize=(8, 6))

    for class_index, class_name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(
            y_true_bin[:, class_index],
            y_score[:, class_index]
        )

        roc_auc = auc(fpr, tpr)

        plt.plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"{class_name} (AUC = {roc_auc:.3f})"
        )

    # Diagonal random-classifier reference line
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Random classifier"
    )

    plt.title(f"{model_name}: ROC Curves for Each Class")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)


    output_path = FIG_DIR / f"{model_name}_roc_curves.png"
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("ROC curve figure saved to:", output_path)

# Select the best model from the final comparison table
best_model_name = summary_df.sort_values(
    by="Test Accuracy (%)",
    ascending=False
).iloc[0]["Model"]

print("Best model selected for ROC curves:", best_model_name)

best_y_true = results[best_model_name]["y_true"]
best_y_score = results[best_model_name]["probabilities"]

plot_multiclass_roc_curves(
    y_true=best_y_true,
    y_score=best_y_score,
    class_names=class_names,
    model_name=best_model_name
)

# 14_ (P) Final model accuracy and loss with fine-tuning phase

This figure shows the full training behaviour of the best-performing model.  
The vertical dashed line indicates where fine-tuning starts.  
Before this point, only the classification head is trained. After this point, selected pretrained layers are unfrozen and trained using a smaller learning rate.

This helps show whether the model generalises well, because the validation accuracy and validation loss can be compared against the training curves.

In [ ]:
def plot_final_model_training_summary(history, model_name, test_accuracy=None, fine_tune_start_epoch=None):
    """
    Plot final model training summary:
    1. Training and validation accuracy
    2. Training and validation loss
    3. Optional fine-tuning phase marker
    4. Optional final test accuracy in the title
    """

    if hasattr(history, "history"):
        history_data = history.history
    else:
        history_data = history

    train_acc = np.array(history_data["accuracy"]) * 100
    val_acc = np.array(history_data["val_accuracy"]) * 100
    train_loss = history_data["loss"]
    val_loss = history_data["val_loss"]

    epochs = list(range(1, len(train_acc) + 1))

    plt.figure(figsize=(13, 5))

    # Accuracy subplot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_acc, marker="o", label="Training Accuracy")
    plt.plot(epochs, val_acc, marker="^", label="Validation Accuracy")

    if fine_tune_start_epoch is not None:
        plt.axvline(
            x=fine_tune_start_epoch,
            linestyle="--",
            label="Fine-tuning start"
        )

    plt.title("Training and Validation Accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Loss subplot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_loss, marker="o", label="Training Loss")
    plt.plot(epochs, val_loss, marker="^", label="Validation Loss")

    if fine_tune_start_epoch is not None:
        plt.axvline(
            x=fine_tune_start_epoch,
            linestyle="--",
            label="Fine-tuning start"
        )

    plt.title("Training and Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)

    if test_accuracy is not None:
        title_text = f"{model_name}: Final Model Test Set Accuracy = {test_accuracy * 100:.2f}%"
    else:
        title_text = f"{model_name}: Final Training and Validation Summary"

    plt.suptitle(title_text, fontweight="bold")
    plt.tight_layout()

    output_path = FIG_DIR / f"{model_name}_final_training_summary.png"
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("Final training summary graph saved to:", output_path)


# Select the best-performing model from the final comparison table
best_model_name = summary_df.sort_values(
    by="Test Accuracy (%)",
    ascending=False
).iloc[0]["Model"]

best_model_accuracy = summary_df.loc[
    summary_df["Model"] == best_model_name,
    "Test Accuracy (%)"
].values[0]

# Convert if accuracy is stored as a percentage instead of decimal
if best_model_accuracy > 1:
    best_model_accuracy = best_model_accuracy / 100

print("Best model selected:", best_model_name)
print(f"Best model test accuracy: {best_model_accuracy * 100:.2f}%")

plot_final_model_training_summary(
    history=histories[best_model_name],
    model_name=best_model_name,
    test_accuracy=best_model_accuracy,
    fine_tune_start_epoch=HEAD_EPOCHS
)


## 15. Grad-CAM visual explanation

Grad-CAM is used to make the trained CNN model more explainable.

Instead of only reporting accuracy, Grad-CAM highlights the image regions that influenced the model prediction. This is useful for a medical imaging project because it gives a visual check of whether the model is focusing on meaningful MRI regions rather than irrelevant background areas.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from pathlib import Path
import random

# Grad-CAM visualisation for the best-performing model
# This cell rebuilds the best architecture, reloads its saved weights,
# and generates class-activation heatmaps for test-set MRI examples.

LAST_FEATURE_LAYER = {
    "ResNet50": "conv5_block3_out",
    "DenseNet121": "conv5_block16_concat",
    "EfficientNetB0": "top_activation"
}

def rebuild_trained_model(model_name):
    """Rebuild a model architecture and restore the best saved weights."""
    restored_model, _ = build_model(model_name)
    weight_file = weight_paths[model_name]

    if not Path(weight_file).exists():
        raise FileNotFoundError(
            f"Could not find saved weights for {model_name}: {weight_file}"
        )

    restored_model.load_weights(str(weight_file))
    return restored_model


def find_feature_layer(base_model, model_name):
    """
    Return the preferred final spatial feature layer.
    If the expected layer name is not available, fall back to the last 4-D layer.
    """
    preferred_name = LAST_FEATURE_LAYER.get(model_name)

    if preferred_name is not None:
        try:
            base_model.get_layer(preferred_name)
            return preferred_name
        except ValueError:
            pass

    # Safe fallback: choose the last layer whose output is image-like: H x W x C.
    for layer in reversed(base_model.layers):
        try:
            if len(layer.output.shape) == 4:
                return layer.name
        except Exception:
            continue

    raise ValueError(f"No suitable 4-D feature layer found inside {model_name}.")


def load_test_image(image_path):
    """Load one image from disk using the same size as the training pipeline."""
    image = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    image = tf.keras.utils.img_to_array(image).astype("float32")
    return image


def make_gradcam_heatmap(model, model_name, image_array, target_class=None):
    """
    Generate a Grad-CAM heatmap for one image.

    Parameters
    ----------
    model : keras.Model
        Trained model with a pretrained CNN backbone.
    model_name : str
        Name of the architecture, e.g. ResNet50.
    image_array : np.ndarray
        Image array with shape (224, 224, 3).
    target_class : int or None
        If None, Grad-CAM is calculated for the model's predicted class.

    Returns
    -------
    heatmap : np.ndarray
        A 2-D array scaled between 0 and 1.
    predicted_class : int
        The predicted class index.
    confidence : float
        The predicted class confidence.
    """
    # Get the base model layer using its actual name within the full model
    base_model = model.get_layer(model_name.lower())
    feature_layer_name = find_feature_layer(base_model, model_name)
    feature_layer = base_model.get_layer(feature_layer_name)

    # Model from backbone input to the chosen convolutional feature map.
    feature_reader = keras.Model(
        inputs=base_model.input,
        outputs=feature_layer.output
    )

    # Prepare image for the selected backbone.
    image_batch = np.expand_dims(image_array, axis=0)
    preprocessed_batch = MODEL_REGISTRY[model_name]["preprocess"](image_batch.copy())

    with tf.GradientTape() as tape:
        feature_maps = feature_reader(preprocessed_batch, training=False)
        tape.watch(feature_maps)

        # Reuse the trained classification head on the final spatial features.
        x = model.get_layer("global_pool")(feature_maps)
        x = model.get_layer("feature_projection")(x)
        predictions = model.get_layer("class_output")(x)

        predicted_class = int(tf.argmax(predictions[0]).numpy())
        if target_class is None:
            target_class = predicted_class

        class_score = predictions[:, target_class]

    # Gradient of target score with respect to the feature maps.
    gradients = tape.gradient(class_score, feature_maps)

    # Weight each feature channel by its average gradient importance.
    channel_weights = tf.reduce_mean(gradients, axis=(0, 1, 2))
    feature_maps = feature_maps[0]

    heatmap = tf.reduce_sum(feature_maps * channel_weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0)

    # Normalise safely to [0, 1].
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    heatmap = tf.image.resize(heatmap[..., tf.newaxis], IMG_SIZE)
    heatmap = tf.squeeze(heatmap).numpy()

    confidence = float(predictions[0, predicted_class].numpy())
    return heatmap, predicted_class, confidence


def overlay_gradcam(image_array, heatmap, alpha=0.45):
    """Blend a Grad-CAM heatmap with the original MRI image."""
    heatmap_uint8 = np.uint8(255 * heatmap)
    colour_map = plt.colormaps["jet"](heatmap_uint8)[..., :3]
    colour_map = np.uint8(255 * colour_map)

    original = np.uint8(np.clip(image_array, 0, 255))
    overlay = np.uint8((1 - alpha) * original + alpha * colour_map)
    return overlay


def choose_gradcam_examples(samples_per_class=1):
    """
    Select a small number of test images from each class folder.
    The test folder is used because these images were not used for training.
    """
    selected = []

    for class_index, class_name in enumerate(class_names):
        folder = SPLIT_DIR / "test" / class_name
        candidates = sorted(
            [p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]
        )

        if len(candidates) == 0:
            continue

        random.Random(SEED + class_index).shuffle(candidates)
        for path in candidates[:samples_per_class]:
            selected.append((path, class_index, class_name))

    return selected


def plot_gradcam_gallery(model, model_name, samples_per_class=1):
    """
    Create a gallery with three columns:
    original image, Grad-CAM heatmap, and overlay.
    """
    examples = choose_gradcam_examples(samples_per_class=samples_per_class)
    row_count = len(examples)

    if row_count == 0:
        print("No test images found for Grad-CAM.")
        return

    fig, axes = plt.subplots(row_count, 3, figsize=(11, 3.2 * row_count))

    # Make axes 2-D even when there is only one row.
    if row_count == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, (image_path, true_index, true_name) in enumerate(examples):
        image_array = load_test_image(image_path)
        heatmap, pred_index, confidence = make_gradcam_heatmap(
            model=model,
            model_name=model_name,
            image_array=image_array,
            target_class=None
        )
        overlay = overlay_gradcam(image_array, heatmap)

        predicted_name = class_names[pred_index]

        axes[row, 0].imshow(np.uint8(image_array))
        axes[row, 0].set_title(f"Original\nTrue: {true_name}")

        axes[row, 1].imshow(heatmap, cmap="jet")
        axes[row, 1].set_title("Grad-CAM heatmap")

        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(
            f"Overlay\nPred: {predicted_name} ({confidence:.2%})"
        )

        for col in range(3):
            axes[row, col].axis("off")

    plt.suptitle(f"Grad-CAM visual explanation using {model_name}", fontweight="bold")
    plt.tight_layout()

    output_path = FIG_DIR / f"gradcam_{model_name}.png"
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("Grad-CAM figure saved to:", output_path)


# Use the best model found in the comparison section.
best_model_name = best_model
print("Preparing Grad-CAM for:", best_model_name)

gradcam_model = rebuild_trained_model(best_model_name)
plot_gradcam_gallery(
    model=gradcam_model,
    model_name=best_model_name,
    samples_per_class=1
)


# Generated outputs

Figures and machine-readable outputs produced by the notebook are saved under the working directory.

Saved location:

```text
/content/brain_tumor_mri_work/figures
```

In [ ]:
print("Notebook run complete.")
print("Figures folder:", FIG_DIR)
print("Weights folder:", WEIGHTS_DIR)
print("Results folder:", RESULTS_DIR)
print("Model metrics CSV:", RESULTS_DIR / "model_metrics.csv")

for path in sorted(FIG_DIR.glob("*")):
    print("-", path.name)

## Portfolio interpretation and limitations

- The reported results come from an **image-level split within one public dataset**.
- The split is stratified by class, but it is **not a verified patient-level split**; related slices from the same patient may therefore be a potential source of leakage if the source dataset contains them.
- The experiment does not establish performance across hospitals, scanners or patient populations.
- Grad-CAM is an explainability aid and does not prove clinically meaningful tumour localisation.
- This project is an academic machine-vision prototype and must not be used for diagnosis or treatment decisions.
